In [2]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("processed.csv")

texts = df["text"].fillna("").tolist()
labels = df["label"]

print(df.shape)

(1713, 9)


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.90
)

X_1 = vectorizer.fit_transform(texts)

print(X_1.shape)

(1713, 20000)


In [18]:
from sklearn.model_selection import train_test_split

X_train_1, X_test_1, y_train_1, y_test_1 = train_test_split(
    X_1,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

In [19]:
HIGH_RISK_WORDS = {
    "risk", "uncertainty", "litigation", "default", "bankruptcy",
    "impairment", "loss", "decline", "failure", "adverse", "volatile",
    "exposure", "deficit", "breach", "penalty", "fraud", "disruption",
    "inflation", "recession", "headwind", "challenge", "difficulty"
}

LOW_RISK_WORDS = {
    "growth", "profit", "revenue", "expansion", "opportunity", "strong",
    "positive", "increase", "improvement", "exceed", "dividend",
    "stable", "robust", "efficient", "successful", "innovative",
    "milestone", "record", "gain", "favorable", "confident", "profitable"
}

NEGATIONS = {
    "not", "no", "never", "neither",
    "nor", "none", "cannot", "without"
}

In [20]:
def custom_features(text):
    tokens = text.split()

    n = max(len(tokens), 1)

    high = sum(1 for t in tokens if t in HIGH_RISK_WORDS)
    low = sum(1 for t in tokens if t in LOW_RISK_WORDS)
    neg = sum(1 for t in tokens if t in NEGATIONS)

    avg_word_length = (
        np.mean([len(t) for t in tokens])
        if tokens else 0
    )

    unique_ratio = len(set(tokens)) / n

    sentence_count = max(
        text.count(".") +
        text.count("!") +
        text.count("?"),
        1
    )

    return [
        np.log1p(n),
        high / n,
        low / n,
        high / (low + 1e-6),
        neg / n,
        avg_word_length,
        unique_ratio,
        np.log1p(sentence_count)
    ]

In [21]:
X_custom = np.array(
    [custom_features(text) for text in texts],
    dtype=np.float32
)

print(X_custom.shape)

(1713, 8)


In [22]:
scaler = StandardScaler()

X_custom_scaled = scaler.fit_transform(X_custom)

print(X_custom_scaled.shape)

(1713, 8)


In [23]:
X_2 = sp.hstack(
    [
        X_1,
        sp.csr_matrix(X_custom_scaled)
    ],
    format="csr"
)

print(X_2.shape)

(1713, 20008)


In [24]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X_2,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

print(X_train_2.shape)
print(X_test_2.shape)

(1370, 20008)
(343, 20008)


In [25]:
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")
joblib.dump(scaler, "custom_scaler.pkl")

['custom_scaler.pkl']

In [27]:
from sentence_transformers import SentenceTransformer

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "C:\Users\M HARI TEJA\miniconda3\envs\env\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
sbert_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
embeddings = sbert_model.encode(
    texts,
    batch_size=16,
    show_progress_bar=True
)

print(embeddings.shape)

In [ ]:
X_3 = embeddings

In [ ]:
X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    X_3,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

In [ ]:
joblib.dump(
    "sentence-transformers/all-MiniLM-L6-v2",
    "sbert_model_name.pkl"
)

In [28]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [32]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(labels)

print(le.classes_)

['high_risk' 'low_risk' 'medium_risk']


In [93]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_test_2 = le.fit_transform(y_test_2)
y_train_2 = le.fit_transform(y_train_2)

In [29]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

In [34]:
xgb_model.fit(X_train_1, y_train_1)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, objective='multi:softprob', ...)

In [35]:
xgb_model2 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)
xgb_model2.fit(X_train_2,le.fit_transform( y_train_2))

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, objective='multi:softprob', ...)

In [39]:
test_acc_xgb1 = (
    xgb_model.predict(X_test_1) ==  y_test_1
).mean()

print(test_acc_xgb1)

0.6793002915451894


In [40]:
test_acc_xgb2 = (
    xgb_model2.predict(X_test_2) == y_test_2
).mean()

print(test_acc_xgb2)

0.7580174927113703


In [78]:
# ada_model = AdaBoostClassifier(
#     estimator=DecisionTreeClassifier(max_depth=3),
#     n_estimators=200,
#     learning_rate=0.5,
#     algorithm="SAMME",
#     random_state=42
# )
ada_model2 = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1),
    n_estimators=50,
    algorithm="SAMME",
    random_state=42

)

In [79]:
# X_train_dense_1 = X_train_1.toarray()
# X_test_dense_1 = X_test_1.toarray()
X_train_dense_2 = X_train_2.toarray()
X_test_dense_2 = X_test_2.toarray()

In [48]:
# ada_model.fit(
#     X_train_dense_1,
#     le.fit_transform(y_train_1)
# )

In [81]:
ada_model2.fit(
    X_train_dense_2,
    y_train_2
)

AdaBoostClassifier(algorithm='SAMME',
                   estimator=DecisionTreeClassifier(max_depth=1),
                   random_state=42)

In [ ]:
# test_acc_ada1 = (
#     ada_model.predict(X_test_dense_1) ==le.fit_transform( y_test_1)
# ).mean()

# print(test_acc_ada1)

In [82]:
test_acc_ada2 = (
    ada_model2.predict(X_test_dense_2) ==y_test_2
).mean()

print(test_acc_ada2)

0.7288629737609329


In [83]:
train_acc_ada2 = (
    ada_model2.predict(X_train_dense_2) ==s y_train_2
).mean()

print(train_acc_ada2)

0.7423357664233576


In [ ]:
# cat_model1 = CatBoostClassifier(
#     iterations=300,
#     depth=6,
#     learning_rate=0.1,
#     loss_function="MultiClass",
#     eval_metric="Accuracy",
#     verbose=0
# )

# cat_model1.fit(
#     X_train_1,
#     le.fit_transform(y_train_1)
# )

In [94]:
cat_model2 = CatBoostClassifier(
    iterations=100,
    depth=4,
    learning_rate=0.1,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    verbose=0
)

cat_model2.fit(
    X_train_2.astype(np.float32),
    y_train_2
)

In [ ]:
# test_acc_cat1 = (
#     cat_model1.predict(X_train_1).flatten() ==le.fit_transform( y_test_1)
# ).mean()

# print(test_acc_cat1)

In [95]:
train_acc_cat2 = (
    cat_model2.predict(X_train_2.astype(np.float32)).flatten() == y_train_2
).mean()

print(train_acc_cat2)

0.8437956204379562


In [96]:
test_acc_cat2 = (
    cat_model2.predict(X_test_2.astype(np.float32)).flatten() == y_test_2
).mean()

print(test_acc_cat2)

0.7230320699708455


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [72]:
print(type(y_test_2.iloc[0]) if hasattr(y_test_2, "iloc") else type(y_test_2[0]))
print(xgb_model2.predict(X_test_2)[:5])

<class 'str'>
[0 1 0 1 1]


In [97]:
from sklearn.metrics import classification_report

print("XGBoost")
print(classification_report(y_test_2, xgb_model2.predict(X_test_2)))

print("CatBoost")
print(classification_report(
    y_test_2,
    cat_model2.predict(X_test_2.astype(np.float32)).flatten()
))
print("AdaBoost")
print(classification_report(
    y_test_2,
    ada_model2.predict(X_test_2.astype(np.float32)).flatten()
))

XGBoost
              precision    recall  f1-score   support

           0       0.80      0.84      0.82       117
           1       0.83      0.79      0.81       113
           2       0.65      0.65      0.65       113

    accuracy                           0.76       343
   macro avg       0.76      0.76      0.76       343
weighted avg       0.76      0.76      0.76       343

CatBoost
              precision    recall  f1-score   support

           0       0.78      0.79      0.79       117
           1       0.79      0.77      0.78       113
           2       0.60      0.60      0.60       113

    accuracy                           0.72       343
   macro avg       0.72      0.72      0.72       343
weighted avg       0.72      0.72      0.72       343

AdaBoost
              precision    recall  f1-score   support

           0       0.87      0.71      0.78       117
           1       0.83      0.71      0.77       113
           2       0.57      0.77      0.66      